### Welcome to Week 6 Day 3!

Let's experiment with a bunch more MCP Servers

In [30]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

True

### The first type of MCP Server: runs locally, everything local

Here's a really interesting one: a knowledge-graph based memory.

It's a persistent memory store of entities, observations about them, and relationships between them.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory


In [14]:
# Use npx with full path and absolute database path
# IMPORTANT: Use absolute path to ensure consistency across all cells
import os
# Get the absolute path to ensure consistency - use current working directory
# which should be the notebook's directory when running in Jupyter
base_dir = os.path.abspath(os.getcwd())
memory_db_path = os.path.abspath(os.path.join(base_dir, "memory", "ed.db"))
print(f"Base directory: {base_dir}")
print(f"Database path: {memory_db_path}")
print(f"Database exists: {os.path.exists(memory_db_path)}")
# Ensure PATH includes /usr/local/bin where node is located
current_path = os.environ.get("PATH", "/usr/local/bin:/usr/bin:/bin")
if "/usr/local/bin" not in current_path:
    current_path = "/usr/local/bin:" + current_path
memory_params = {
    "command": "/usr/local/bin/npx",
    "args": ["-y", "mcp-memory-libsql"],
    "env": {"LIBSQL_URL": "file:" + memory_db_path, "PATH": current_path}
}
print(f"LIBSQL_URL: file:{memory_db_path}")

async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

# The following tools allow to create conectivity among the things we want to remember using entities and relations

Base directory: /Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/6_mcp
Database path: /Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/6_mcp/memory/ed.db
Database exists: True
LIBSQL_URL: file:/Users/federico.tognetti/Desktop/AgenticAIEngineering/agents/6_mcp/memory/ed.db


[Tool(name='create_entities', description='Create new entities with observations', inputSchema={'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'entityType': {'type': 'string'}, 'observations': {'type': 'array', 'items': {'type': 'string'}}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities'], '$schema': 'http://json-schema.org/draft-07/schema#'}, annotations=None, title='Create new entities with observations'),
 Tool(name='search_nodes', description='Search for entities and their relations using text search with relevance ranking', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string'}, 'limit': {'type': 'number'}}, 'required': ['query'], '$schema': 'http://json-schema.org/draft-07/schema#'}, annotations=None, title='Search for entities and their relations using text search with relevance ranking'),
 Tool(name='read_graph', description='Get recent entit

In [15]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Ed. I'm an LLM engineer. I'm teaching a course about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4.1-mini"

In [16]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

# The comand will also create the file ed.db in the memory folder

Nice to meet you, Ed! It's great that you are teaching a course about AI Agents and covering the MCP protocol. If you want, I can help you by providing explanations, creating teaching materials, or answering questions related to AI Agents and MCP. How would you like to proceed?

In [17]:
async with MCPServerStdio(params=memory_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Ed. What do you know about me?")
    display(Markdown(result.final_output))

I know that you are Ed. You are an LLM engineer, you are teaching a course about AI Agents, and you are knowledgeable about the MCP protocol. Is there anything specific you would like to add or update about yourself?

### Check the trace:

https://platform.openai.com/traces

### The 2nd type of MCP server - runs locally, calls a web service

### Brave Search - apologies - this will need another API key! But it's free again.

https://brave.com/search/api/

Set up your account, and put your key in the .env under `BRAVE_API_KEY`

In [ ]:
env = {"BRAVE_API_KEY": os.getenv("BRAVE_API_KEY")}
# params = {"command": "npx", "args": ["-y", "@modelcontextprotocol/server-brave-search"], "env": env}

# Ensure PATH includes /usr/local/bin where node is located
current_path = os.environ.get("PATH", "/usr/local/bin:/usr/bin:/bin")
if "/usr/local/bin" not in current_path:
    current_path = "/usr/local/bin:" + current_path
# Add PATH to the environment
env["PATH"] = current_path
brave_params = {
    "command": "/usr/local/bin/npx",
    "args": ["-y", "@modelcontextprotocol/server-brave-search"],
    "env": env
}

async with MCPServerStdio(params=brave_params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

# This code:
# 1. Uses npx to run the "@modelcontextprotocol/server-brave-search" MCP server locally (the package is downloaded if not present).
# 2. Launches the server as a subprocess and sets the required environment variables, including your BRAVE_API_KEY, for Brave Search API access.
# 3. Ensures the system PATH includes /usr/local/bin so node/npx can be found.
# 4. Lists the available MCP tools provided by the running Brave Search MCP server.


[Tool(name='brave_web_search', description='Performs a web search using the Brave Search API, ideal for general queries, news, articles, and online content. Use this for broad information gathering, recent events, or when you need diverse web sources. Supports pagination, content filtering, and freshness controls. Maximum 20 results per request, with offset for pagination. ', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query (max 400 chars, 50 words)'}, 'count': {'type': 'number', 'description': 'Number of results (1-20, default 10)', 'default': 10}, 'offset': {'type': 'number', 'description': 'Pagination offset (max 9, default 0)', 'default': 0}}, 'required': ['query']}, annotations=None),
 Tool(name='brave_local_search', description="Searches for local businesses and places using Brave's Local Search API. Best for queries related to physical locations, businesses, restaurants, services, etc. Returns detailed information including:\

In [21]:
instructions = "You are able to search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. \
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4o-mini"

In [22]:
async with MCPServerStdio(params=brave_params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

As of February 10, 2026, Amazon's stock price is experiencing volatility. Here are the key highlights:

1. **Current Price**: The latest closing stock price for Amazon (AMZN) was $210.28 on February 6, 2026, following a recent drop attributed to a broader tech sell-off.

2. **Recent Decline**: The stock fell nearly 8.8%, dropping from a previous close of around $222.69 to approximately $203.15 due to concerns regarding a $200 billion capital expenditure plan that is perceived to be costly in the AI sector.

3. **Market Sentiment**: Despite the recent sell-off, some analysts believe that this dip could set the stage for a rebound later in the year, citing positive news from CEO Andy Jassy regarding upcoming strategies and growth prospects.

4. **Forecasts**: Predictions for Amazon's stock price vary, with some analysts suggesting a potential recovery while others project a decrease in price, anticipating a drop to around $186 by the end of February 2026.

### Outlook
Overall, while Amazon faces significant headwinds in the short term due to market uncertainties and heavy investments in AI, there is cautious optimism about a potential rebound later in 2026, contingent on positive earnings and growth in its AWS division.

### As usual, check out the trace:

https://platform.openai.com/traces

## And now the third type: running remotely

It's actually really hard to find a "remote MCP server" aka "hosted MCP server" aka "managed MCP server".

It's not a common model for using or sharing MCP servers, and there isn't a standard way to discover remote MCP servers.

Anthropic lists some remote MCP servers, but these are for paid applications with business users:

https://docs.anthropic.com/en/docs/agents-and-tools/remote-mcp-servers

CloudFlare has tooling for you to create and deploy your own remote MCP servers, but this does not seem to be a common practice:

https://developers.cloudflare.com/agents/guides/remote-mcp-server/


# And back to the 2nd type: the Polygon.io MCP Server

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/stop.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">PLEASE READ!!-</h2>
            <span style="color:#ff7800;">This service for financial market data has both a FREE plan and a PAID plan, and we can use either depending on your appetite.
            </span>
        </td>
    </tr>
</table>

## NEW SECTION: Introducing polygon.io

Polygon.io is a hugely popular financial data provider. It has a free plan and a paid plan. And it also has an MCP Server!

First, read up on polygon.io on their excellent website, including looking at their pricing:

https://polygon.io

### Polygon.io Part 1: Polygon.io free service (the paid will be totally optional, of course!)

1. Please sign up for polygon.io (top right)  
2. Once signed in, please select "Keys" in the left hand navigation
3. Press the blue "New Key" button
4. Copy the key name
5. Edit your .env file and add the row:

`POLYGON_API_KEY=xxxx`

In [ ]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY is not set")

In [ ]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

### Wrapped into a python module that caches end of day prices

I've made a python module `market.py` that uses this API to look up share prices.

But the free API is quite heavily rate limited - so I've been a bit sneaky; when you ask for a share price, this function retrieves the entire end-of-day equity market, and caches it in our database.


In [ ]:
from market import get_share_price
get_share_price("AAPL")

In [ ]:
# no rate limiting concerns!

for i in range(1000):
    get_share_price("AAPL")
get_share_price("AAPL")

### And I've made this into an MCP Server

Just as we did with accounts.py; see `market_server.py`

In [ ]:
params = {"command": "uv", "args": ["run", "market_server.py"]}
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools

### Let's try it out!

Hopefully gpt-4o-mini is smart enough to know that the symbol for Apple is AAPL

In [ ]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple?"
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Polygon.io Part 2: Paid Plan - Totally Optional!

If you are interested, you can subscribe to the monthly plan to get more up to date market data, and unlimited API calls.

If you do wish to do this, then it also makes sense to use the full MCP server that Polygon.io has released, to take advantage of all their functionality.



In [ ]:

params = {"command": "uvx",
          "args": ["--from", "git+https://github.com/polygon-io/mcp_polygon@v0.1.0", "mcp_polygon"],
          "env": {"POLYGON_API_KEY": polygon_api_key}
          }
async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()
mcp_tools


### Wow that's a lot of tools!

Let's try them out - hopefully the sheer number of tools doesn't overwhelm gpt-4o-mini!

With the $29 monthly plan, we don't have access to some of the APIs, so I've needed to specify which APIs can be called.

If you've splashed out on a bigger plan, feel free to remove my extra constraint..

In [ ]:
instructions = "You answer questions about the stock market."
request = "What's the share price of Apple? Use your get_snapshot_ticker tool to get the latest price."
model = "gpt-4.1-mini"

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

## Setting up your .env file

If you do decide to have a paid plan, please add this to your .env file to indicate:

`POLYGON_PLAN=paid`

And if you decide to go all the way for the realtime API, then please do:

`POLYGON_PLAN=realtime`

In [ ]:
load_dotenv(override=True)

polygon_plan = os.getenv("POLYGON_PLAN")
is_paid_polygon = polygon_plan == "paid"
is_realtime_polygon = polygon_plan == "realtime"

if is_paid_polygon:
    print("You've chosen to subscribe to the paid Polygon plan, so the code will look at prices on a 15 min delay")
elif is_realtime_polygon:
    print("Wowzer - you've chosen to subscribe to the realtime Polygon plan, so the code will look at realtime prices")
else:
    print("According to your .env file, you've chosen to subscribe to the free Polygon plan, so the code will look at EOD prices")

## And that's it for today!

I've removed the part of this lab that uses the "Financial Datasets" mcp server, because it's inferior - more expensive with fewer APIs.

And this way we get to use the same provider for Free and Paid APIs.

But if you want to see the code, just look in the git history for a prior version.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercises</h2>
            <span style="color:#ff7800;">Explore MCP server marketplaces and integrate your own, using all 3 approaches.
            </span>
        </td>
    </tr>
</table>

In [34]:
# This is a test to import and connect to an MCP server from github
# IMPORTANT: excel-mcp-server is available as a Python package via uvx
# Use the format: uvx excel-mcp-server stdio (no --from needed)
# Ensure PATH includes /usr/local/bin where node is located (needed for any Node.js dependencies)
import os
current_path = os.environ.get("PATH", "/usr/local/bin:/usr/bin:/bin")
if "/usr/local/bin" not in current_path:
    current_path = "/usr/local/bin:" + current_path

params = {
    "command": "/Users/federico.tognetti/.local/bin/uvx",
    "args": ["excel-mcp-server", "stdio"],
    "env": {"PATH": current_path}
}

async with MCPServerStdio(params=params, client_session_timeout_seconds=60) as server:
    mcp_tools = await server.list_tools()

mcp_tools


[Tool(name='apply_formula', description='\n    Apply Excel formula to cell.\n    Excel formula will write to cell with verification.\n    ', inputSchema={'properties': {'filepath': {'title': 'Filepath', 'type': 'string'}, 'sheet_name': {'title': 'Sheet Name', 'type': 'string'}, 'cell': {'title': 'Cell', 'type': 'string'}, 'formula': {'title': 'Formula', 'type': 'string'}}, 'required': ['filepath', 'sheet_name', 'cell', 'formula'], 'title': 'apply_formulaArguments', 'type': 'object'}, annotations=None, outputSchema={'properties': {'result': {'title': 'Result', 'type': 'string'}}, 'required': ['result'], 'title': 'apply_formulaOutput', 'type': 'object'}),
 Tool(name='validate_formula_syntax', description='Validate Excel formula syntax without applying it.', inputSchema={'properties': {'filepath': {'title': 'Filepath', 'type': 'string'}, 'sheet_name': {'title': 'Sheet Name', 'type': 'string'}, 'cell': {'title': 'Cell', 'type': 'string'}, 'formula': {'title': 'Formula', 'type': 'string'}},